# 8 awk and sed

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part II — Pipelines and text extraction</span>
    <span class="bp-meta">Notebook&nbsp;8</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    The two tools that finish the toolkit: sed edits a stream, and awk is
    field-aware and can compute, clearing the two walls the last notebook hit.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v1.0.1&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; every sed edit works on a fresh scratch copy.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

Notebook 7 ended at two walls. `tr` could only swap characters, not replace
*strings*. `cut` could only split on a *single* character, and could not do
arithmetic. These two tools clear both walls and finish the extraction toolkit:

- **`sed`** edits a stream (find-and-replace and more), picking up the regular
  expressions from Notebook 6.
- **`awk`** splits each line into fields and can **compute** on them, which is the
  climax of Part II: doing actual arithmetic on the numbers in real data.

A word of honesty first. `sed` and `awk` are each whole programming languages;
people write hundred-line programs in them. We are not doing that. We take the
**daily workhorse slice** of each, name the deep parts as out of scope, and stop
there. That is the right amount for a shell primer, and it is genuinely most of
what you will ever type. (And, as always: averaging a column of energies is just
averaging a column of numbers: **no physics required.**)

## A. `sed` — stream editing

`sed` reads its input line by line, applies an editing command to each, and prints
the result. The mental model is simple: **it is `grep` that can also change what it
finds.** Its headline command is substitution, `s/old/new/`.

```{command-card} sed
```

Here it is on a scratch config: replace every `300` with `500`. Note `sed` prints
to the screen and leaves the file untouched (more on that below):

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'TEMPERATURE 300\nCUTOFF 300\nMAX_STEPS 1000\n# a trailing comment\n' > scratch/run.in

In [3]:
sed 's/300/500/g' scratch/run.in

TEMPERATURE 500


CUTOFF 500


MAX_STEPS 1000


# a trailing comment


The `g` made it global (every match on each line, not just the first). The regular
expression is the **same language as Notebook 6** — reach for `sed -E` to get the
Extended syntax, since `sed`, like `grep`, defaults to the quirkier Basic regex.
Two separate things, worth not conflating: `-E` buys the Extended *pattern*
syntax, while `&` in the *replacement* stands for the whole matched text in
both dialects — `sed 's/[0-9][0-9]*/<&>/'` works with no `-E` at all. It is
handy for wrapping or annotating a match:

In [4]:
sed -E 's/[0-9]+/<&>/g' scratch/run.in

TEMPERATURE <300>


CUTOFF <300>


MAX_STEPS <1000>


# a trailing comment


`sed` can also just *select*. With `-n` (don't auto-print) and the `p` command, it
prints only matching lines, essentially `grep`:

In [5]:
sed -n '/CUTOFF/p' scratch/run.in

CUTOFF 300


And `d` deletes. Addressed by a regex, `/^#/d` strips comment lines; addressed by
number, `2,4d` deletes a line range:

In [6]:
sed '/^#/d' scratch/run.in

TEMPERATURE 300


CUTOFF 300


MAX_STEPS 1000


The full `sed` command vocabulary is below. Beyond it lie the hold space,
branches, and multi-command scripts: real, formidable, and firmly **out of scope**
for this course. When you need them, you will know, and you will reach for a
reference.

<div class="bp-card">
  <span class="bp-card-cmd">sed commands</span> — <span class="bp-card-job">the workhorse slice (with <code>-E</code> for ERE). The deep parts (hold space, branching) are out of scope.</span>
  <table>
    <tr><td>s/old/new/</td><td>substitute the first match on each line</td></tr>
    <tr><td>s/old/new/g</td><td>substitute every match on the line</td></tr>
    <tr><td>s/old/new/2</td><td>substitute only the 2nd match; add <code>I</code> for case-insensitive</td></tr>
    <tr><td>&amp;</td><td>in the replacement, the whole matched text</td></tr>
    <tr><td>-n  /pat/p</td><td>print only lines matching pat (like grep)</td></tr>
    <tr><td>/pat/d   2,5d   $d</td><td>delete: lines matching pat / lines 2–5 / the last line</td></tr>
  </table>
</div>

:::{admonition} ⚠ `sed -i` edits in place — no undo
:class: danger
Everything above printed to the screen and left the file alone. Add **`-i`** and
`sed` rewrites the file *in place*, and, like `rm` and `>`, with **no undo**. A
substitution with a slightly-too-greedy pattern, run with `-i`, has quietly mangled
many a config. Two habits, the same family as the earlier warnings: dry-run
*without* `-i` first and read the output, and when you do commit, use **`-i.bak`**,
which saves the original as `file.bak` before editing. Never `sed -i` something you
have not first seen the result of.
:::

In [7]:
sed -i.bak 's/300/500/g' scratch/run.in

In [8]:
ls scratch

run.in	run.in.bak


The edit happened, and `run.in.bak` holds the original: your safety net.

## B. `awk` — fields and computation

`awk` also reads line by line, but it does something `sed` does not: it **splits
each line into fields**, `$1`, `$2`, … up to `$NF` (the last), and lets you act on
them. Crucially, it splits on *any run of whitespace* by default, which is exactly
the wall `cut` hit. Where `cut -d' '` miscounts ragged spacing, `awk` just works:

```{command-card} awk
```

In [9]:
grep -m1 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | awk '{print $NF}'

-142.246533543175843


`$NF` pulled the energy (the last field) no matter how many spaces padded the
line. That single fact retires `cut` for anything real. The structure of an `awk`
program is `pattern { action }`: for every line matching `pattern`, run `action`.
Either part is optional. A bare condition filters; a bare action runs on every
line. `NR` is the current line number, so `NR==1` selects the first line; a
field condition works the same way, so `$1=="Kr"` selects the krypton row:

In [10]:
printf 'Ar 1.5 2.5\nAr 3.5 4.5\nKr 5.5 6.5\n' | awk '$1=="Kr" {print $2, $3}'

5.5 6.5


Now the payoff, and the reason `awk` is the climax of Part II: it can **compute**.
A `BEGIN` block runs before the first line, an `END` block after the last, and in
between you can accumulate. Summing and averaging a column is the canonical move:
here, the six energies from the log:

In [11]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | awk '{ sum += $1; n++ } END { printf "mean = %.4f over %d values\n", sum/n, n }'

mean = -143.4444 over 6 values


That is real arithmetic on real numbers, in one line: the thing no tool before
this notebook could do. The workhorse `awk` vocabulary is below; arrays,
user-defined functions, and `getline` are the **out-of-scope** deep end.

<div class="bp-card">
  <span class="bp-card-cmd">awk programs</span> — <span class="bp-card-job">the workhorse slice. Arrays, functions, and getline are out of scope. Always single-quote the program.</span>
  <table>
    <tr><td>$0   $1 … $NF</td><td>the whole line; field 1 … the last field (NF = field count)</td></tr>
    <tr><td>NR</td><td>the current record (line) number</td></tr>
    <tr><td>pattern { action }</td><td>run action on each line matching pattern (either part optional)</td></tr>
    <tr><td>/regex/   $3&gt;0   NR==1</td><td>patterns: a regex, or a condition on a field or NR</td></tr>
    <tr><td>print   printf</td><td>output a line, or formatted output</td></tr>
    <tr><td>BEGIN{…} END{…}</td><td>run once before the first / after the last line: where totals live</td></tr>
  </table>
</div>

```{admonition} Single-quote your programs
:class: note
Always wrap a `sed` or `awk` program in **single** quotes: `awk '{print $1}'`. Those
`$1`, `$2` look exactly like shell positional parameters, and in double quotes the
shell would replace them before `awk` ever ran, usually with nothing. This is the
same "the shell goes first" rule from globbing and `grep`, and it is the single most
common `awk`/`sed` mistake.
```

## Exercises

A full set, climbing to the canonical trajectory task. Every `sed` edit works on a
fresh `scratch/` copy (`data/` is never touched), and the notebook gives the same
result every run.

### Warm-up 1 (worked) — `sed` substitute and select

On a scratch copy: globally substitute, then use `-n '/pat/p'` to print only
matching lines.

In [12]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'alpha 1\nbeta 2\nalpha 3\ngamma 4\n' > scratch/tags.txt

In [13]:
# (solution hidden on the public site)


ALPHA 1


beta 2


ALPHA 3


gamma 4


beta 2


In [14]:
check '[ "$(sed "s/alpha/ALPHA/g" scratch/tags.txt | grep -c ALPHA)" -eq 2 ] && [ "$(sed -n "/beta/p" scratch/tags.txt)" = "beta 2" ]' \
      "both alphas were upper-cased and only the beta line was selected"

✓ both alphas were upper-cased and only the beta line was selected


### Warm-up 2 (your turn) — `sed` delete by address

On a scratch copy, delete the comment lines (those starting with `#`) with
`/^#/d`, and separately delete the first two lines with `1,2d`.

In [15]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf '# header comment\nkeep one\nkeep two\n# mid comment\nkeep three\n' > scratch/doc.txt

In [16]:
# (solution hidden on the public site)


keep one


keep two


keep three


keep two


# mid comment


keep three


In [17]:
check '[ "$(sed "/^#/d" scratch/doc.txt | grep -c "^#")" -eq 0 ] && [ "$(sed "/^#/d" scratch/doc.txt | wc -l)" -eq 3 ] && [ "$(sed "1,2d" scratch/doc.txt)" = "$(tail -n 3 scratch/doc.txt)" ]' \
      "the two comment lines are gone, and 1,2d left exactly the last three lines"

✓ the two comment lines are gone, and 1,2d left exactly the last three lines


### Applied 1 (your turn) — `awk` fields clear the wall

The energy line is padded with ragged spaces, so `cut -d' '` can't reliably grab a
column. Show that, then pull the energy (the **last** field) with `awk`.

In [18]:
cd "$ROOT"

In [19]:
# (solution hidden on the public site)


(a.u.):


-142.246533543175843


In [20]:
got="$(grep -m1 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | awk '{print $NF}')"
check '[ "$got" = "-142.246533543175843" ]' \
      "awk pulled the energy field where cut could not"

✓ awk pulled the energy field where cut could not


### Applied 2 (your turn) — `awk` patterns

From the trajectory's comment lines, print only the frames whose energy (the last
field) is below `-0.06`. (Pattern: a condition on `$NF`.)

In [21]:
cd "$ROOT"

In [22]:
# (solution hidden on the public site)


91, -0.0601988256


94, -0.0604130301


97, -0.0609348186


100, -0.0610322312


103, -0.0613274365


...

175, -0.0621023237


178, -0.0621027034


181, -0.0621033425


184, -0.0621037945


186, -0.0621038220


In [23]:
below="$(grep -E '^[[:space:]]+i =' data/trajectories/lj38-optimization.xyz | awk '$NF < -0.06' | wc -l)"
notbelow="$(grep -E '^[[:space:]]+i =' data/trajectories/lj38-optimization.xyz | awk '$NF >= -0.06' | wc -l)"
check '[ "$below" -gt 0 ] && [ "$notbelow" -gt 0 ]' \
      "the condition split the frames into below and not-below -0.06"

✓ the condition split the frames into below and not-below -0.06


### Applied 3 (worked) — `awk` arithmetic

Sum and average a column with an `END` block: the six log energies.

In [24]:
cd "$ROOT"

In [25]:
# (solution hidden on the public site)


sum=-860.6663 mean=-143.4444


In [26]:
mean="$(grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | awk '{s+=$1;n++} END{printf "%.4f", s/n}')"
check '[ "$mean" = "-143.4444" ]' \
      "awk computed the mean energy, -143.4444"

✓ awk computed the mean energy, -143.4444


### Composite — putting it together (the canonical task)

The move you came to Part II for. From a real trajectory, extract the energy of
every frame (the comment line's last field), then compute the **mean** energy, the
**minimum**, and the **frame** at which the minimum occurs, all in one `awk` pass
over the lines `grep` selects. Composes `grep` (NB6), `awk`, and the pipe (NB4).

In [27]:
cd "$ROOT"

In [28]:
# (solution hidden on the public site)


mean = -0.054455


min  = -0.062104  (frame 186)


In [29]:
out="$(grep -E '^[[:space:]]+i =' data/trajectories/lj38-optimization.xyz | awk '{e=$NF; sum+=e; n++; if(n==1||e<min){min=e; fmin=$3+0}} END{printf "%.6f %.6f %d", sum/n, min, fmin}')"
check '[ "$out" = "-0.054455 -0.062104 186" ]' \
      "mean -0.054455, minimum -0.062104, occurring at frame 186"

✓ mean -0.054455, minimum -0.062104, occurring at frame 186


### Optional stretch (your turn) — Safe in-place edit

Do a real in-place edit, the safe way. On a scratch copy, use `sed -i.bak` to make
a substitution, then confirm the `.bak` backup holds the original and the file
holds the change. The point is the *habit*: never `-i` without a backup you have
checked.

In [30]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'cutoff = 300\n' > scratch/p.in

In [31]:
# (solution hidden on the public site)


cutoff = 500


cutoff = 300


In [32]:
check 'grep -q 500 scratch/p.in && grep -q 300 scratch/p.in.bak' \
      "the edit took effect in the file, and the .bak preserved the original"

✓ the edit took effect in the file, and the .bak preserved the original


## Outlook

You can now extract *and compute* from any text file the course holds, by hand,
one one-liner at a time. That last phrase is the whole point of what comes next:
you have been **retyping** these pipelines. Part III is about making them
*reusable* (writing and editing them as scripts you keep), and it starts with the
one tool you still lack: a way to write text in the terminal. Next (Notebook 9):
**editing with Vim.**

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself; nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions (to teach from or to check your own work), get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>